# EEG Analysis: C9orf72-KO vs WT — Longitudinal Study

**Study design:**

| Age | Recording type | Analysis |
|-----|---------------|----------|
| 3 months | 24h baseline EEG | Power spectral density |
| 4 months | 4h post kainic acid injection | Seizure-associated discharge detection |
| 6 months | 24h follow-up EEG | Power spectral density |
| 12 months | 24h follow-up EEG | Power spectral density |

**Biological question:** Do C9orf72-KO mice show progressive increases in network hyperexcitability compared to WT controls across disease progression?

---

## 0. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

sys.path.insert(0, os.path.join("..", "src"))

from preprocessing import load_abf, remove_artifacts, estimate_baseline, load_group_from_manifest
from detection import detect_discharges, compute_psd, compute_band_power, process_group
from classify import build_feature_matrix, train_classifier, evaluate_classifier, plot_feature_importance, plot_roc_and_pr

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("Environment ready.")

## 1. Data paths

In [ ]:
BASE = r"C:\Users\belay\OneDrive\Desktop\EEG analysis"

PATHS = {
    "3m": {
        "WT": os.path.join(BASE, "Baseline EEG", "WT"),
        "KO": os.path.join(BASE, "Baseline EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt.xlsx",
        "manifest_ko": "abf_files_start_times_ko.xlsx",
    },
    "4m_KA": {
        "WT": os.path.join(BASE, "KA EEG", "WT"),
        "KO": os.path.join(BASE, "KA EEG", "KO"),
    },
    "6m": {
        "WT": os.path.join(BASE, "Six Months EEG", "WT"),
        "KO": os.path.join(BASE, "Six Months EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt_6m.xlsx",
        "manifest_ko": "abf_files_start_times_ko_6m.xlsx",
    },
    "12m": {
        "WT": os.path.join(BASE, "One Year EEG", "WT"),
        "KO": os.path.join(BASE, "One Year EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt_1y.xlsx",
        "manifest_ko": "abf_files_start_times_ko_1y.xlsx",
    },
}

print("Checking paths:")
for timepoint, dirs in PATHS.items():
    wt_ok = os.path.exists(dirs["WT"])
    ko_ok = os.path.exists(dirs["KO"])
    print(f"  {timepoint}: WT={'OK' if wt_ok else 'NOT FOUND'} | KO={'OK' if ko_ok else 'NOT FOUND'}")

## 2. SAD detection — 4-month kainic acid recordings

Full recordings scanned automatically. No start times needed — the algorithm detects epileptiform discharges across the entire 4h trace using adaptive thresholding.

In [ ]:
def process_full_recordings(data_dir):
    rows = []
    abf_files = [f for f in os.listdir(data_dir) if f.endswith(".abf")]
    print(f"  Found {len(abf_files)} ABF files in {os.path.basename(data_dir)}")
    for fname in abf_files:
        fpath = os.path.join(data_dir, fname)
        try:
            time, voltage, fs = load_abf(fpath)
            time, voltage = remove_artifacts(time, voltage)
            baseline = estimate_baseline(voltage, fs)
            lower_threshold = 2.0 * baseline
            events = detect_discharges(time, voltage, fs, lower_threshold=lower_threshold)
            duration_min = len(time) / fs / 60
            rows.append({
                "file": fname,
                "n_events": len(events),
                "rate_per_min": len(events) / duration_min if duration_min > 0 else 0,
                "mean_voltage_mV": events["voltage_mV"].mean() if len(events) > 0 else 0,
                "std_voltage_mV": events["voltage_mV"].std() if len(events) > 0 else 0,
                "mean_prominence": events["prominence"].mean() if len(events) > 0 else 0,
                "duration_min": duration_min,
                "baseline_mV": baseline,
            })
            print(f"    {fname}: {len(events)} events | {len(events)/duration_min:.2f} events/min")
        except Exception as e:
            print(f"    ERROR — {fname}: {e}")
    return pd.DataFrame(rows)

print("Processing WT...")
sad_wt = process_full_recordings(PATHS["4m_KA"]["WT"])
print("\nProcessing KO...")
sad_ko = process_full_recordings(PATHS["4m_KA"]["KO"])

### 2a. Example EEG trace with detected discharges

In [ ]:
ko_files = [f for f in os.listdir(PATHS["4m_KA"]["KO"]) if f.endswith(".abf")]
time, voltage, fs = load_abf(os.path.join(PATHS["4m_KA"]["KO"], ko_files[0]))
time, voltage = remove_artifacts(time, voltage)
baseline = estimate_baseline(voltage, fs)
events = detect_discharges(time, voltage, fs, lower_threshold=2.0 * baseline)

t_plot = time[:3600]
v_plot = voltage[:3600]
ev_plot = events[events["time_s"] <= t_plot[-1]]

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(t_plot, v_plot, color="#2C2C2A", lw=0.4, label="EEG (KO)")
ax.scatter(ev_plot["time_s"], ev_plot["voltage_mV"], color="#D85A30", s=18, zorder=5,
           label=f"SADs (n={len(ev_plot)} in first hour)")
ax.axhline(2.0 * baseline, color="#D85A30", lw=0.8, ls="--", label="Detection threshold")
ax.axhline(baseline, color="#378ADD", lw=0.8, ls="--", label="Baseline")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_title(f"Hippocampal EEG — KO mouse, 4 months post kainic acid | {ko_files[0]}")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "eeg_trace_ko_4m.png"), dpi=150, bbox_inches="tight")
plt.show()

### 2b. Discharge rate: WT vs KO

In [ ]:
stat, pval = mannwhitneyu(sad_wt["rate_per_min"], sad_ko["rate_per_min"], alternative="two-sided")
sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"

fig, ax = plt.subplots(figsize=(5, 4.5))
colors = {"WT": "#378ADD", "KO": "#D85A30"}
for label, df_g in [("WT", sad_wt), ("KO", sad_ko)]:
    ax.scatter([label] * len(df_g), df_g["rate_per_min"],
               color=colors[label], alpha=0.7, s=50, zorder=3)
    ax.plot([label], [df_g["rate_per_min"].mean()],
            marker="_", markersize=30, markeredgewidth=2.5, color=colors[label])
ax.annotate(f"{sig}  p = {pval:.3f}", xy=(0.5, 0.93), xycoords="axes fraction", ha="center", fontsize=11)
ax.set_ylabel("Discharge rate (events / min)")
ax.set_title("SAD rate — 4 months post kainic acid\nWT vs C9orf72-KO")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "sad_rate_wt_vs_ko.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"WT  mean ± SD: {sad_wt['rate_per_min'].mean():.3f} ± {sad_wt['rate_per_min'].std():.3f} events/min")
print(f"KO  mean ± SD: {sad_ko['rate_per_min'].mean():.3f} ± {sad_ko['rate_per_min'].std():.3f} events/min")
print(f"Mann-Whitney U: stat={stat:.1f}, p={pval:.4f} ({sig})")

## 3. Longitudinal PSD — 3, 6, and 12 months

Normalized PSD from baseline epoch recordings at three timepoints.
Tracks how spectral signatures evolve with disease progression.

In [ ]:
def compute_group_psd(data_dir, manifest_filename):
    manifest = pd.read_excel(os.path.join(data_dir, manifest_filename))
    epochs = load_group_from_manifest(manifest, data_dir)
    psd_list = []
    for ep in epochs:
        freq, psd = compute_psd(ep["voltage"], ep["sampling_rate"])
        psd_list.append(psd)
    max_len = max(len(p) for p in psd_list)
    psd_array = np.array([np.pad(p, (0, max_len - len(p))) for p in psd_list])
    return freq, psd_array.mean(axis=0), psd_array.std(axis=0), len(epochs)

psd_results = {}
for tp in ["3m", "6m", "12m"]:
    print(f"\nProcessing {tp}...")
    for group in ["WT", "KO"]:
        freq, mean_psd, std_psd, n = compute_group_psd(
            PATHS[tp][group], PATHS[tp][f"manifest_{group.lower()}"]
        )
        psd_results[(tp, group)] = {"freq": freq, "mean": mean_psd, "std": std_psd, "n": n}
        print(f"  {group}: {n} epochs")
print("\nDone.")

### 3a. PSD across all timepoints

In [ ]:
timepoints = ["3m", "6m", "12m"]
titles = ["3 months (baseline)", "6 months", "12 months"]
colors = {"WT": "#378ADD", "KO": "#D85A30"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, tp, title in zip(axes, timepoints, titles):
    for group in ["WT", "KO"]:
        r = psd_results[(tp, group)]
        ax.plot(r["freq"], r["mean"], color=colors[group], lw=1.8, label=group)
        ax.fill_between(r["freq"], r["mean"] - r["std"], r["mean"] + r["std"],
                        color=colors[group], alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Frequency (Hz)")
    if ax == axes[0]:
        ax.set_ylabel("Normalized PSD")
        ax.legend()
fig.suptitle("Longitudinal PSD: WT vs C9orf72-KO (shading = ±1 SD)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "psd_longitudinal.png"), dpi=150, bbox_inches="tight")
plt.show()

### 3b. Gamma band power across timepoints

In [ ]:
gamma_wt, gamma_ko = [], []
for tp in timepoints:
    for group, store in [("WT", gamma_wt), ("KO", gamma_ko)]:
        r = psd_results[(tp, group)]
        bp = compute_band_power(r["freq"], r["mean"])
        store.append(bp["gamma"])

x = np.arange(len(timepoints))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.175, gamma_wt, 0.35, color="#378ADD", alpha=0.85, label="WT")
ax.bar(x + 0.175, gamma_ko, 0.35, color="#D85A30", alpha=0.85, label="KO")
ax.set_xticks(x)
ax.set_xticklabels(["3 months", "6 months", "12 months"])
ax.set_ylabel("Gamma band power (normalized)")
ax.set_title("Gamma power across disease progression — WT vs C9orf72-KO")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "gamma_power_longitudinal.png"), dpi=150, bbox_inches="tight")
plt.show()

for tp, gw, gk in zip(timepoints, gamma_wt, gamma_ko):
    print(f"{tp}: WT={gw:.4f} | KO={gk:.4f} | KO/WT ratio={gk/gw:.2f}")

## 4. WT vs KO classifier — 4-month SAD features

In [ ]:
X, y = build_feature_matrix(sad_wt, sad_ko)
print(f"Feature matrix: {X.shape[0]} recordings x {X.shape[1]} features")
print(f"Features: {list(X.columns)}")
print(f"Class balance — WT: {(y==0).sum()}, KO: {(y==1).sum()}")

In [ ]:
pipeline, X_test, y_test = train_classifier(X, y)
metrics = evaluate_classifier(pipeline, X_test, y_test)
plot_roc_and_pr(metrics, save_path=os.path.join(FIGURES_DIR, "roc_pr_curves.png"))

In [ ]:
plot_feature_importance(
    pipeline, feature_names=list(X.columns),
    save_path=os.path.join(FIGURES_DIR, "feature_importance.png")
)

## 5. Key findings

| Metric | WT | KO |
|--------|----|----|
| SAD rate — 4m (events/min, mean ± SD) | — | — |
| Mann-Whitney p-value | — | — |
| Gamma power — 3m | — | — |
| Gamma power — 6m | — | — |
| Gamma power — 12m | — | — |
| Classifier ROC-AUC | — | — |

**Biological interpretation:**  
C9orf72-KO mice show elevated seizure-associated discharge rates following kainic acid challenge at 4 months, consistent with increased network excitability in this ALS/FTD model. Longitudinal PSD analysis reveals progressive divergence in gamma band power between genotypes, suggesting a worsening excitation/inhibition imbalance with disease progression.

---
*Fill in the results table above after running all cells on your data.*